# 数值数据处理综合实践

学习目标：把多通道模拟测量整理为可检查的数值结果，处理无效值与常量列，完成排序、导出和重复运行核对。

前置知识：索引、广播、统计聚合、随机采样、数组读写、函数与数值验证。

运行环境：Python 3.12、NumPy 2.5.3；随机输入使用 PCG64 与固定种子。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

全部输入为自制模拟数据，不代表真实设备。后续单元沿用首次导入的 np；导出文件在临时目录中往返，结束后自动清理；配套原始 CSV 保留。

配套脚本：本章无外部脚本；数据位于 date/。

（1）[测量 CSV](date/19-measurements.csv)：三个通道的自制读数，单位为任意单位，NA 表示缺测，inf 为非有限观测。

## 1 从小型测量表开始

下面四行表示四次观测，三列表示三个通道。先根据有限值计算各通道均值：本例将 NaN 和正负无穷都视为无效观测，保留它们的位置但不参与统计。

isfinite() 识别有限数值，where() 生成以 NaN 标记无效位置的新数组；nanmean() 忽略 NaN，沿观测轴统计。这个无效值口径是本任务的约定，不是对所有数据集通用的清洗规则。

In [1]:
import numpy as np

raw = np.array([[1.0, 10.0, 5.0], [3.0, np.nan, 5.0], [5.0, 14.0, 5.0], [np.inf, np.nan, 5.0]])
valid = np.isfinite(raw)
clean = np.where(valid, raw, np.nan)
means = np.nanmean(clean, axis=0, keepdims=True)

print(clean.shape, clean.dtype)  # (4, 3) float64，清洗保留二维结构。
print(valid.sum(axis=0))  # [3 2 4]，各通道有效数量。
print(means)  # [[3. 12. 5.]]，均值分母分别为 3、2、4。
print(raw[3, 0])  # inf，原数组没有被修改。

(4, 3) float64
[3 2 4]
[[ 3. 12.  5.]]
inf


## 2 先明确处理规则

这次实践只处理二维实数测量数组，并采用以下规则。规则先于计算确定，不能在看到结果后随意修改。

（1）至少有一行和一列；通道编号数量必须与列数相同。

（2）每列至少有一次有限观测。全缺失列不属于本例输入范围，不能把空统计解释为有效的零。

（3）按每列有效观测的均值与 ddof=0 标准差（方差分母为本列有效观测数 N，即 N−ddof）缩放；这是对当前数据本身的摘要，不声称估计总体参数。

（4）有效观测逐值相等的常量列，有效位置记为 0，无效位置仍为 NaN。其他列使用“减均值、除标准差”。用有限值的最小值与最大值精确相等来识别常量，不依赖计算出的标准差恰好为零；近似常量需要另定业务阈值。

以下沿用上一单元的 raw，查看通道编号、有效数量和缺失坐标。

In [2]:
channel_ids = np.array([101, 102, 103], dtype=np.int64)
# 每列对应一个通道；axis=0 汇总该通道的各次观测。
counts = np.isfinite(raw).sum(axis=0)
print(channel_ids, counts)  # 三个通道与有效数量 [3 2 4] 一一对应。
print(np.nonzero(~np.isfinite(raw)))  # 无效位置的行索引与列索引，按坐标配对。

[101 102 103] [3 2 4]
(array([1, 3, 3]), array([1, 0, 1]))


## 3 缩放并保留缺失位置

继续使用 clean。keepdims=True 让均值与标准差保持 (1, 3)，与四行测量广播。np.divide() 的 where 只计算有效的非常量位置，out 预先填 NaN，未计算位置不会留下未初始化内容。

先用 nanmin 与 nanmax 的精确比较识别常量。再把每列减去它的有限最小值，得到 shifted；常量列的有效值于是精确为 0，无效位置仍为 NaN。均值偏移 mean_offsets 与标准差都在这个平移后的坐标中计算，缩放时也使用 shifted − mean_offsets，避免先恢复一个较大的均值再相减。

这一改写可由均值与标准差定义直接推导：对一列有效值 x，令 c 为该列最小值、y = x − c，则 mean(y) = mean(x) − c，且 y − mean(y) = x − mean(x)，所以精确算术中的标准差不变。浮点运算不保证不同写法逐位相同；这里的平移减少了较大共同偏置对近邻值差异的影响。

返回的原尺度均值由 minimum + mean_offsets 恢复，可能舍入，不能保证用这个摘要重新计算出完全相同的缩放结果。有效常量位置单独赋 0，这表示没有相对偏差，不是用零填补缺失。

In [3]:
# 1. 沿观测轴识别常量列，再在平移坐标中计算均值和标准差。
minimum = np.nanmin(clean, axis=0, keepdims=True)
constant = minimum == np.nanmax(clean, axis=0, keepdims=True)
shifted = clean - minimum
mean_offsets = np.nanmean(shifted, axis=0, keepdims=True)
means = minimum + mean_offsets
scales = np.nanstd(shifted, axis=0, ddof=0, keepdims=True)
# 2. 输出先填 NaN，只计算有效且非常量的位置；常量有效值另设为 0。
scaled = np.full(clean.shape, np.nan, dtype=np.float64)
np.divide(shifted - mean_offsets, scales, out=scaled, where=valid & ~constant)
scaled = np.where(valid & constant, 0.0, scaled)

print(means, scales)  # 均值 [3, 12, 5]；标准差约 [1.63299316, 2, 0]。
print(scaled)
# 首列约 [-1.22474487, 0, 1.22474487, nan]；第二列 [-1, nan, 1, nan]。
# 第三列为四个 0；没有把原无效位置替换成有效值。
print(np.array_equal(np.isnan(scaled), ~valid))  # True，缺失位置保持。
print(np.all(np.isfinite(scaled[valid])))  # True，有效位置仍为有限数值。

[[ 3. 12.  5.]] [[1.63299316 2.         0.        ]]
[[-1.22474487 -1.          0.        ]
 [ 0.                 nan  0.        ]
 [ 1.22474487  1.          0.        ]
 [        nan         nan  0.        ]]
True
True


## 4 手算与数值检查

首列有效值为 1、3、5，均值 3，ddof=0 方差为 8/3；第二列有效值 10、14，均值 12、标准差 2；第三列全部为 5。

把这些独立手算结果写成参考数组，再检查形状、类型、缺失位置和容差。本例都是小型、适度量级 float64 计算，使用 rtol=0、atol=1e-12；equal_nan=True 只用于确认缺失位置对应，不把 NaN 当作正常观测。

In [4]:
expected = np.array([
    [-2 / np.sqrt(8 / 3), -1.0, 0.0],
    [0.0, np.nan, 0.0],
    [2 / np.sqrt(8 / 3), 1.0, 0.0],
    [np.nan, np.nan, 0.0],
])
np.testing.assert_allclose(scaled, expected, rtol=0, atol=1e-12, equal_nan=True, strict=True)
print("手算结果核对通过")
print(np.nanmean(scaled, axis=0))  # 三列均接近 0。
print(np.nanstd(scaled, axis=0, ddof=0))  # [1. 1. 0.]，常量列仍为常量。

手算结果核对通过
[0. 0. 0.]
[1. 1. 0.]


## 5 按通道读数排序

按第一通道的清洗后读数升序排列所有观测，使用同一个排序索引重排行编号与缩放结果。NaN 排在末尾，stable=True 保持相等键的原顺序。

排序改变了行位置，因此同时保留原观测编号。下面继续使用 clean 与 scaled。

In [5]:
observation_ids = np.array([1, 2, 3, 4], dtype=np.int64)
order = np.argsort(clean[:, 0], stable=True)
sorted_ids = observation_ids[order]
sorted_scaled = scaled[order]

print(sorted_ids)  # [1 2 3 4]，本例有效读数本来有序，无效键放末尾。
print(sorted_scaled.shape)  # (4, 3)，所有通道一起随行移动。
print(np.array_equal(sorted_scaled, scaled[order], equal_nan=True))  # True。

[1 2 3 4]
(4, 3)
True


## 6 将已讲步骤组合为函数

输入为非空二维实数数组，每列至少有一个有限值。返回缩放结果、均值、标准差与有效数量；通道编号用于标识列，不参与数值计算。

本例使用可转换为 float64 的适度量级数据。整数超出 −2**53 至 2**53 时，转换可能丢失相邻整数的差；极大差值可能溢出，极小差值的平方可能下溢。输入若已在上游被舍入，函数无法恢复丢失的信息。遇到这些情况应先调整数据单位或选择适合的算法。

函数直接组合前面的计算步骤，不重复检查已说明的输入条件。原尺度均值仅作摘要，标准化继续使用平移坐标。

In [6]:
def summarize_channels(values):
    """缩放本章二维测量，返回结果、列均值、列标准差和有效数量。"""
    # 1. 每行是一次观测、每列是一个通道；非有限值统一记为 NaN。
    data = np.asarray(values, dtype=np.float64)
    valid = np.isfinite(data)
    counts = valid.sum(axis=0)
    clean = np.where(valid, data, np.nan)

    # 2. 列统计保留 (1, 通道数)，便于向所有观测行广播。
    minimum = np.nanmin(clean, axis=0, keepdims=True)
    constant = minimum == np.nanmax(clean, axis=0, keepdims=True)
    shifted = clean - minimum
    offsets = np.nanmean(shifted, axis=0, keepdims=True)
    means = minimum + offsets
    scales = np.nanstd(shifted, axis=0, ddof=0, keepdims=True)

    # 3. where 未选中的位置保留 out 中的 NaN，常量列仅将有效位置设为 0。
    result = np.full(clean.shape, np.nan, dtype=np.float64)
    np.divide(shifted - offsets, scales, out=result, where=valid & ~constant)
    result = np.where(valid & constant, 0.0, result)
    return result, means, scales, counts


result, means, scales, counts = summarize_channels(raw)
print(np.allclose(result, expected, rtol=0, atol=1e-12, equal_nan=True))  # True，与手算一致。
print(counts, result.shape)  # [3 2 4] (4, 3)。

True
[3 2 4] (4, 3)


## 7 边界输入

### 7.1 识别输入范围

本例要求每列至少有一个有限观测。下面第二列没有有效值，不能把它的统计量解释为零；若任务需要保留这种列，必须另行约定输出，见篇末练习。

In [7]:
data = np.array([[1.0, np.nan], [2.0, np.inf]])
counts = np.isfinite(data).sum(axis=0)
print(data.shape, counts)  # (2, 2)，有效数量为 [2, 0]。
print(counts == 0)  # [False, True]：第二列超出本例函数的输入范围。

(2, 2) [2 0]
[False  True]


### 7.2 常量列中仍有缺失

某列的所有有效观测都相同，但部分行缺失时，仍只把有效位置缩放为 0。下面第一列两个有效值均为 5，第二列有三个不同的值。

这项检查可以识别错误写法“整列都填 0”，它会误把缺失位置变成有效观测。

In [8]:
data = np.array([[5.0, 1.0], [np.nan, 2.0], [5.0, 3.0]])
result, means, scales, counts = summarize_channels(data)

print(result[:, 0])  # [0. nan 0.]，中间缺失不变。
print(scales, counts)  # 标准差约 [0, 0.8165]；有效数量 [2 3]。
np.testing.assert_array_equal(np.isnan(result), ~np.isfinite(data))
print("缺失位置核对通过")  # 预期：常量列的缺失掩码保持不变，断言通过后显示此提示。

[ 0. nan  0.]
[[0.         0.81649658]] [2 3]
缺失位置核对通过


### 7.3 舍入残差与数值退化

重复的 0.1 在 float64 中是同一个值，应该归为常量；对它求均值和标准差仍可能产生微小舍入残差。有限值是否逐值相等与计算出的标准差是否为零是不同判断。

下面先观察常量归零，再直接观察大整数转换与极小差值平方。这里展示浮点表示的限制；这些数据在数学上仍可有均值和标准差，但超出了本例计算所约定的尺度。

整数输入可无损转成 float64，也不代表其均值能精确表示。下面两个相邻大整数的均值落在它们之间；1.0 与朝 2.0 方向的相邻浮点数也有相同问题。nextafter 给出指定方向的相邻可表示浮点数。两个不同观测在 ddof=0 下距均值各半个间隔，因此本例缩放结果应为 −1、1；用平移坐标保留这一差异。

In [9]:
# 常量的有效位置归零，缺失位置仍保留。
decimal_constant = np.array([[0.1], [0.1], [np.nan], [0.1]])
result, means, scales, counts = summarize_channels(decimal_constant)
np.testing.assert_array_equal(result[:, 0], [0.0, 0.0, np.nan, 0.0])
np.testing.assert_array_equal(scales, [[0.0]])
print(result[:, 0], means, scales)  # 常量有效位置全为 0，均值为 0.1，缺失保持。

# 转成 float64 后，相邻大整数可能合并为同一个值。
large = np.array([2**53, 2**53 + 1], dtype=np.int64)
print(np.diff(large.astype(np.float64)))  # [0.]，原本相差 1 的信息已经丢失。
# 极小的非零差值平方可能下溢为 0，不表示原数据是常量。
tiny = np.array([1e-200, 2e-200])
print((tiny - tiny.min()) ** 2)  # [0. 0.]。

# 在可表示的相邻输入上观察平移计算，不用舍入后的均值重新相减。
for label, values in [
    ("相邻大整数", np.array([[2**53 - 1], [2**53]], dtype=np.int64)),
    ("相邻浮点数", np.array([[1.0], [np.nextafter(1.0, 2.0)]])),
]:
    result, means, scales, counts = summarize_channels(values)
    np.testing.assert_allclose(result[:, 0], [-1.0, 1.0], rtol=0, atol=1e-12)
    print(label, result[:, 0], scales)  # 都为 [-1, 1]；标准差为半个间隔。
    print(means)  # 原尺度均值会舍入；结果并未用这个舍入后的摘要再次相减。

[ 0.  0. nan  0.] [[0.1]] [[0.]]
[0.]
[0. 0.]
相邻大整数 [-1.  1.] [[0.5]]
[[9.00719925e+15]]
相邻浮点数 [-1.  1.] [[1.11022302e-16]]
[[1.]]


## 8 可复现的模拟输入

用 PCG64、种子 2026、一次 normal 调用生成六行三列的数据，再按固定位置加入缺失、无穷和常量列。这些修改也是输入生成过程的一部分。

重复实验要保留 NumPy 版本、位生成器、种子、调用顺序与参数；本章使用 NumPy 2.5.3，不承诺跨版本、构建或机器逐位一致。函数在固定原始输入上重复处理，应得到相同结果；这是可复现性，不要求把已经缩放的数据再处理一遍仍完全不变。

In [10]:
rng = np.random.Generator(np.random.PCG64(2026))
simulated = rng.normal(loc=10.0, scale=2.0, size=(6, 3))
simulated[1, 0] = np.nan
simulated[4, 1] = np.inf
simulated[:, 2] = 5.0
first, means, scales, counts = summarize_channels(simulated)
second, _, _, _ = summarize_channels(simulated)

print(simulated)  # 六行三列模拟输入；两处无效，第三列固定为 5。
print(counts, first.shape, first.dtype)  # [5 5 6] (6, 3) float64。
print(np.array_equal(first, second, equal_nan=True))  # True，同一输入重复处理一致。
print(np.nanmean(first, axis=0))  # 各列在舍入误差内接近 0。
print(np.nanstd(first, axis=0, ddof=0))  # 前两列接近 1，第三列为 0。

[[ 8.41375505 10.48114257  5.        ]
 [        nan 11.27658948  5.        ]
 [ 9.37610134 10.60767074  5.        ]
 [ 9.54818228 11.44013564  5.        ]
 [ 9.87174413         inf  5.        ]
 [ 8.77196321  9.19249947  5.        ]]
[5 5 6] (6, 3) float64
True
[-8.8817842e-17  4.4408921e-17  0.0000000e+00]
[1. 1. 0.]


## 9 导出与读回

将缩放结果、通道编号和摘要保存在同一个 npz 文件中。这里全部是数值数组，不需要 pickle；读取时明确 allow_pickle=False。npz 通过 with 关闭，外层 TemporaryDirectory 在结束时清理文件。

二进制往返应保持形状、dtype、数值和 NaN 位置，因此这里采用精确数组断言；这与对计算误差使用容差是两种不同要求。

In [11]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory(prefix="numpy-measurements-") as directory:
    output_path = Path(directory) / "measurements.npz"
    np.savez(output_path, scaled=first, channels=np.array([101, 102, 103]), means=means, scales=scales, counts=counts)
    with np.load(output_path, allow_pickle=False) as saved:
        restored = saved["scaled"]
        np.testing.assert_array_equal(restored, first, strict=True)
        np.testing.assert_array_equal(saved["counts"], counts, strict=True)
        print(saved.files)  # scaled、channels、means、scales、counts 五个数组。
        print(restored.shape, restored.dtype)  # (6, 3) float64。
        print("文件往返核对通过")
print(output_path.exists())  # False，资源关闭后临时文件已清理。

['scaled', 'channels', 'means', 'scales', 'counts']


(6, 3) float64
文件往返核对通过
False


### 9.1 从配套 CSV 读取并处理

先用文本编辑器打开配套文件，辨认表头、NA 和 inf。genfromtxt 按逗号分列，跳过表头，将约定的缺测标记填为 NaN。inf 仍被解析为无穷值，由 summarize_channels 按本章口径排除。

继续使用已定义的函数：三个通道的有效数量分别为 3、2、3；第三列是重复的 0.1。处理后将结果导出到临时 CSV 并读回，原始输入文件不被覆盖。

In [12]:
file_values = np.genfromtxt(
    "date/19-measurements.csv", delimiter=",", skip_header=1,
    missing_values="NA", filling_values=np.nan, dtype=np.float64, encoding="utf-8",
)
file_result, file_means, file_scales, file_counts = summarize_channels(file_values)
print(file_counts)  # [3 2 3]：第二通道只有 10、14 两个有效观测。
print(file_result[:, 2])  # [0. 0. 0. nan]：常量有效位置归零，缺失保留。
np.testing.assert_array_equal(file_counts, [3, 2, 3])
with TemporaryDirectory(prefix="numpy-csv-") as directory:
    csv_output = Path(directory) / "scaled.csv"
    np.savetxt(csv_output, file_result, delimiter=",", fmt="%.18e")
    csv_restored = np.loadtxt(csv_output, delimiter=",")
    np.testing.assert_allclose(csv_restored, file_result, rtol=0, atol=1e-12, equal_nan=True)
    print("CSV 读入、处理与导出核对通过")  # 预期：CSV 读入结果与既有计算一致，断言通过后显示此提示。

[3 2 3]


[ 0.  0.  0. nan]
CSV 读入、处理与导出核对通过


## 本章小结

（1）先写清轴含义、有效值规则和统计口径，再进行广播、聚合与缩放。

（2）空输入、全缺失和常量列需要不同策略；缺失位置与真实零值不能混淆。

（3）手算小样本、边界输入、重复处理与文件往返分别检查不同问题，不能互相替代。

（4）本例提供当前数据的数值摘要；用于预测任务时，统计量的拟合数据范围还须由任务另行约定。

## 练习

（1）手算下方两列的均值、ddof=0 标准差和缩放结果，再调用函数核对。明确每列的有效数量。

In [13]:
data = np.array([[2.0, 10.0], [4.0, np.nan], [6.0, 14.0]])

# 在此写下手算结果，再调用 summarize_channels 并选择容差检查。
# 检查：均值为 [4, 12]，有效数量 [3, 2]；缺失仍处于第 1 行第 1 列。

（2）原函数以每列至少有一个有限值为前提。现在任务要求保留该列，输出和摘要均用 NaN，同时保留有效数量 0。说明必须调整哪一步，并实现一个新的处理版本；不能仅屏蔽警告。

In [14]:
data = np.array([[1.0, np.nan], [3.0, np.inf]])

# 在此先解释新旧约定，再实现新版本；不要覆盖正文函数。
# 提示：先分出有效数量大于 0 的列，只对这些列统计，其他位置预填 NaN。
# 检查：输出保持 (2, 2)，第二列全 NaN，counts 为 [2, 0]。

（3）先预测常量列中的 NaN 是否被替换，再运行。说明该列的零值与缺失分别表示什么。

In [15]:
data = np.array([[7.0, 2.0], [np.nan, 4.0], [7.0, 6.0]])

# 先写下第一列的输出预测，再观察实际结果。
result, means, scales, counts = summarize_channels(data)
print(result[:, 0])
print(counts)

[ 0. nan  0.]
[2 3]


（4）把模拟输入按第二通道的清洗读数升序排列，同时保留原行编号。导出排序结果与行编号，再读回核对。解释为什么不能分别对各列排序。

In [16]:
data = np.array([[1.0, 30.0], [3.0, 10.0], [5.0, 20.0]])
observation_ids = np.array([201, 202, 203])

# 在此处理、取得同一组行索引并同步重排；编号应为 [202, 203, 201]。
# 使用 TemporaryDirectory 和 np.load 的 with 关闭资源，核对值、shape、dtype。
# 说明逐列独立排序为什么会破坏同一次观测的通道对应关系。

### 重点练习提示

对应第（2）题。先独立完成，再按需要查看提示。

（1）有效数量为 0 的列，应在任何 nanmin、nanmean 等统计之前被分开。

（2）先预填全形状的 NaN 结果与摘要，只把有效列送入原计算流程，再放回相同列位置。

### 重点练习参考解析

对应第（2）题。

新版本保留原来的非空二维与数值尺度约定，只放宽“每列至少一次有限观测”。先用 isfinite 按列计数，令 active 表示计数大于 0 的列。结果预填为与输入同形的 float64 NaN，means、scales 预填为 (1, 列数) 的 NaN。

若存在 active 列，只对这些列调用正文的 summarize_channels，把返回结果、均值和标准差写回 active 对应的列；这样仍沿用平移统计和常量列规则。没有有效列时直接保留预填值，不对空列集合做统计。counts 始终使用最初对全部列的计数。

本题结果两行为 [−1, NaN]、[1, NaN]；means 为 [[2, NaN]]，scales 为 [[1, NaN]]，counts 为 [2, 0]。再用全部非有限的 (2, 2) 输入核对：输出与摘要全为 NaN、计数全为 0，形状不变。单纯屏蔽警告既没有限定统计输入，也不能表达新约定。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| NumPy 官方文档（NumPy 2.5） | [isfinite](https://numpy.org/doc/2.5/reference/generated/numpy.isfinite.html)、[where](https://numpy.org/doc/2.5/reference/generated/numpy.where.html) 的有限判断和条件选择；[nanmin](https://numpy.org/doc/2.5/reference/generated/numpy.nanmin.html)、[nanmax](https://numpy.org/doc/2.5/reference/generated/numpy.nanmax.html) 的忽略 NaN 极值；[nanmean](https://numpy.org/doc/2.5/reference/generated/numpy.nanmean.html)、[nanstd](https://numpy.org/doc/2.5/reference/generated/numpy.nanstd.html) 的有效数量、axis、keepdims、ddof 与全缺失边界；[mean](https://numpy.org/doc/2.5/reference/generated/numpy.mean.html)、[std](https://numpy.org/doc/2.5/reference/generated/numpy.std.html) 的 Notes：算术均值、标准差定义与浮点精度限制（正文平移恒等式由这些定义直接推导）；[nextafter](https://numpy.org/doc/2.5/reference/generated/numpy.nextafter.html) 的相邻可表示值；[divide](https://numpy.org/doc/2.5/reference/generated/numpy.divide.html) 的 out、where；[sort](https://numpy.org/doc/2.5/reference/generated/numpy.sort.html) 的稳定性及 NaN 末尾规则、[argsort](https://numpy.org/doc/2.5/reference/generated/numpy.argsort.html) 的间接排序；[Generator.normal](https://numpy.org/doc/2.5/reference/random/generated/numpy.random.Generator.normal.html) 的 loc、scale、size 与[随机流兼容策略](https://numpy.org/doc/2.5/reference/random/compatibility.html)；[savez](https://numpy.org/doc/2.5/reference/generated/numpy.savez.html)、[load](https://numpy.org/doc/2.5/reference/generated/numpy.load.html) 的具名数组、pickle 与关闭资源；[genfromtxt](https://numpy.org/doc/2.5/reference/generated/numpy.genfromtxt.html) 的 delimiter、missing_values、filling_values、skip_header；[savetxt](https://numpy.org/doc/2.5/reference/generated/numpy.savetxt.html)、[loadtxt](https://numpy.org/doc/2.5/reference/generated/numpy.loadtxt.html) 的文本往返；[assert_allclose](https://numpy.org/doc/2.5/reference/generated/numpy.testing.assert_allclose.html)、[assert_array_equal](https://numpy.org/doc/2.5/reference/generated/numpy.testing.assert_array_equal.html) 的容差、strict 与 NaN。有效值、常量列与输入范围是本章明确给出的任务约定。 |
| Python 官方文档（Python 3.12） | [浮点运算限制](https://docs.python.org/3.12/tutorial/floatingpoint.html) 的 53 位精度、表示误差与运算舍入；[TemporaryDirectory](https://docs.python.org/3.12/library/tempfile.html#tempfile.TemporaryDirectory) 的上下文管理和清理；[pathlib 基本使用](https://docs.python.org/3.12/library/pathlib.html#basic-use) 的路径连接和 exists。 |